In [1]:
import os

### Write ```Dockerfile```

In [2]:
%%writefile Dockerfile

FROM python:3.9
    
# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# copy script into container
COPY script.py .

# run script when image is run
CMD ["python3", "script.py"]

Writing Dockerfile


### Write ```requirements.txt``` to local drive

In [3]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

pandas==1.2.4
boto3

Writing requirements.txt


### Write ```script.py``` to local drive

In [4]:
%%writefile script.py

import os
import pandas as pd
import numpy as np
from datetime import datetime
import boto3

# constants
str_project = '20231010-gen-xii'
str_task = 'ad_hoc'
str_subtask = 'gen_11_payload_parsing'
str_final_task = 'concatenate_parsed_payloads'

# get today's date
str_date_today = datetime.today().strftime('%Y%m%d')

# init
cls_client = boto3.client('s3')

# prefix
str_prefix = f'{str_task}/{str_subtask}/days/{str_date_today}/parsed_payloads'

# list the files
dict_response = cls_client.list_objects_v2(
    Bucket=str_project,
    Prefix=str_prefix,
)

# get contents
list_dict_contents = dict_response['Contents']

# get the filenames
list_str_files = [dict_contents['Key'] for dict_contents in list_dict_contents]

# get only gzip
list_str_files = [str_file for str_file in list_str_files if '.gzip' in str_file]
print(f'There are {len(list_str_files)} parsed files:')
for a, str_file in enumerate(list_str_files):
    print(f'{a+1} - {str_file}')

# import and concatenate
list_df = []
for str_file in list_str_files:
    # read
    str_uri = f's3://{str_project}/{str_file}'
    df = pd.read_parquet(str_uri)
    # append
    list_df.append(df)
# concat
df = pd.concat(list_df)
# save memory
del list_df

# write to s3
str_filename = 'df_concat.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_subtask}/days/{str_date_today}/{str_final_task}/{str_filename}'
df.to_parquet(str_uri, compression='gzip')

Writing script.py


### Build and push to ECR

In [5]:
%%sh

# name the image
image=genxi-concat-parsed-payloads

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  26.11kB
Step 1/7 : FROM python:3.9
 ---> ab7eeae5d25f
Step 2/7 : RUN apt-get update
 ---> Using cache
 ---> be6db65d342c
Step 3/7 : RUN pip install --upgrade pip
 ---> Using cache
 ---> 1da7040878d3
Step 4/7 : COPY requirements.txt .
 ---> Using cache
 ---> abc85fead7b7
Step 5/7 : RUN pip install -r requirements.txt
 ---> Using cache
 ---> 023180175e17
Step 6/7 : COPY script.py .
 ---> f5b8710b06e3
Step 7/7 : CMD ["python3", "script.py"]
 ---> Running in 2342549f6379
Removing intermediate container 2342549f6379
 ---> ad3ac7a2ce3e
Successfully built ad3ac7a2ce3e
Successfully tagged genxi-concat-parsed-payloads:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxi-concat-parsed-payloads' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxi-concat-parsed-payloads]
65e86169b10b: Preparing
259a74b4bf4c: Preparing
0c917355e70f: Preparing
1fbb04935245: Preparing
126ef28403f3: Preparing
47e31a4d606a: Preparing
bf4966b4b813: Preparing
da15a2a37253: Preparing
89ca33c95b2e: Preparing
83db175c22e2: Preparing
c5d13b2949a2: Preparing
7e43f593c900: Preparing
072686bcd3db: Preparing
89ca33c95b2e: Waiting
83db175c22e2: Waiting
47e31a4d606a: Waiting
c5d13b2949a2: Waiting
bf4966b4b813: Waiting
da15a2a37253: Waiting
7e43f593c900: Waiting
072686bcd3db: Waiting
259a74b4bf4c: Layer already exists
126ef28403f3: Layer already exists
1fbb04935245: Layer already exists
0c917355e70f: Layer already exists
47e31a4d606a: Layer already exists
89ca33c95b2e: Layer already exists
da15a2a37253: Layer already exists
bf4966b4b813: Layer already exists
83db175c22e2: Layer already exists
c5d13b2949a2: Layer already exists
7e43f593c900: Layer already exists
072686bcd3db: Layer a

### Clean-up

In [6]:
# rm files
for str_file in ['Dockerfile','requirements.txt','script.py']:
    try:
        os.remove(f'./{str_file}')
    except:
        pass